# Disease Prediction Model — SympScan Dataset
Clean, single-purpose notebook. Sirf `Diseases_and_Symptoms_dataset.csv` (SympScan dataset) par kaam kiya gaya hai — pehle wala (itachi) dataset is notebook mein nahi hai.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import ast
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


## 2. Load Dataset

In [2]:
# Dataset load karo
df = pd.read_csv("Diseases_and_Symptoms_dataset.csv")

# Pehla column disease ka naam hai, baaki sab symptom columns (0/1 values)
symptom_columns = df.columns[1:]

df.head()


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,sneezing,leg weakness,hysterical behavior,arm lump or mass,bleeding gums,pain in gums,diaper rash,hesitancy,back stiffness or tightness,low urine output
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


## 3. Quick Data Check
Sirf jo cheezein aage kaam ke liye zaroori hain — shape, missing values, duplicates, disease count.

In [3]:
print("Shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Total diseases:", df["diseases"].nunique())


Shape: (96088, 231)
Missing values: 0
Duplicate rows: 0
Total diseases: 100


# Duplicate & Conflicting Patterns Check

In [16]:
## 3.1 Duplicate Rows Handle Karo

# Duplicate rows drop karo — warna same row train aur test dono mein ja sakti hai,
# jisse test accuracy galat (inflated) lagegi
print("Rows before removing duplicates:", len(df))

df = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates:", len(df))


## 3.2 Conflicting Symptom Patterns Check

# Kabhi-kabhi same symptom combination multiple alag diseases se match karta hai.
# Aise cases model kabhi bhi 100% sahi predict nahi kar sakta — isliye pehle hi pata hona chahiye.

pattern_disease_count = (
    df.groupby(list(symptom_columns), sort=False)["diseases"]
      .nunique()
)

conflicting_patterns = (pattern_disease_count > 1).sum()

print("Total unique symptom patterns:", len(pattern_disease_count))
print("Patterns linked to multiple diseases:", conflicting_patterns)
print(
    f"Estimated max possible accuracy: "
    f"{100 - (conflicting_patterns / len(pattern_disease_count) * 100):.2f}%"
)


## 3.3 Zero-Variance Symptoms Drop Karo

# Jo symptom kabhi 0 hai (kabhi occur hi nahi hota) ya kabhi hamesha 1 hai (sab rows mein hai),
# wo model ke liye koi information nahi deta — bas extra column hai

symptom_counts = df[symptom_columns].sum()

useless_symptoms = symptom_counts[
    (symptom_counts == 0) | (symptom_counts == len(df))
].index.tolist()

print("Useless (zero-variance) symptoms found:", len(useless_symptoms))

df = df.drop(columns=useless_symptoms)
symptom_columns = df.columns[1:]   # update after dropping

print("Remaining symptom columns:", len(symptom_columns))

Rows before removing duplicates: 96088
Rows after removing duplicates: 96088
Total unique symptom patterns: 91210
Patterns linked to multiple diseases: 4046
Estimated max possible accuracy: 95.56%
Useless (zero-variance) symptoms found: 0
Remaining symptom columns: 230


## 4. Train / Test Split
`stratify=y` isliye taaki har disease train aur test dono mein proportionally represent ho.

In [4]:
X = df.drop("diseases", axis=1)
y = df["diseases"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (76870, 230) Test: (19218, 230)


## 5. Label Encoding
Disease names ko numbers mein convert karo, model isi format mein train hota hai.

In [5]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Total classes:", len(label_encoder.classes_))


Total classes: 100


## 6. Train Model

In [6]:
model = LogisticRegression(max_iter=1000, n_jobs=-1)
model.fit(X_train, y_train_encoded)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## 7. Evaluate Model

In [7]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test_encoded, y_pred)
print("Accuracy:", accuracy)

print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))


Accuracy: 0.8985326256634405
                                              precision    recall  f1-score   support

                           actinic keratosis       0.93      0.72      0.81       162
                         acute bronchiolitis       0.95      0.92      0.93       241
                            acute bronchitis       0.87      0.76      0.81       243
                          acute bronchospasm       0.66      0.78      0.71       162
                         acute kidney injury       0.98      0.99      0.98       162
                          acute pancreatitis       0.93      0.85      0.89       241
                             acute sinusitis       0.85      0.92      0.88       166
                                     allergy       0.94      0.98      0.96       161
                                      angina       0.95      0.98      0.96       165
                                     anxiety       0.97      0.97      0.97       240
                        

In [ ]:
## 7.1 Train/Test Overlap Check

# Check karo ki koi exact symptom-pattern train aur test dono mein to nahi hai
train_patterns = set(map(tuple, X_train.values))

overlap_count = sum(
    tuple(row) in train_patterns
    for row in X_test.values
)

print("Test patterns also seen in training:", overlap_count, "out of", len(X_test))

In [8]:
# Top-N accuracy: kitni baar sahi disease top N predictions mein aata hai
def top_n_accuracy(model, X_test, y_test_encoded, n):
    y_proba = model.predict_proba(X_test)
    top_n_indices = np.argsort(y_proba, axis=1)[:, -n:]

    correct = [
        y_test_encoded[i] in top_n_indices[i]
        for i in range(len(y_test_encoded))
    ]
    return np.mean(correct)

print("Top-3 Accuracy:", top_n_accuracy(model, X_test, y_test_encoded, 3))
print("Top-5 Accuracy:", top_n_accuracy(model, X_test, y_test_encoded, 5))


Top-3 Accuracy: 0.9847538765740451
Top-5 Accuracy: 0.9963055468831304


## 8. Save Model
Model, label encoder aur symptom columns save karo — baad mein prediction ke liye reload karenge.

In [9]:
joblib.dump(model, "disease_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
joblib.dump(symptom_columns.tolist(), "symptom_columns.pkl")

print("Saved: disease_model.pkl, label_encoder.pkl, symptom_columns.pkl")


Saved: disease_model.pkl, label_encoder.pkl, symptom_columns.pkl


## 9. Load Supporting Info (description, precautions, diet, workout, medications)
Har file ka disease naam lowercase + strip karke normalize kar rahe hain, taaki lookup consistently match ho.

In [10]:
description_df = pd.read_csv("description.csv")
precautions_df = pd.read_csv("precautions.csv")
diets_df = pd.read_csv("diets.csv")
workout_df = pd.read_csv("workout.csv")
medications_df = pd.read_csv("medications.csv")

# Sab jagah disease naam normalize karo: lowercase + strip
for frame in [description_df, precautions_df, diets_df, workout_df, medications_df]:
    frame["Disease"] = frame["Disease"].str.strip().str.lower()

# precautions.csv mein "copd" hai, model "chronic obstructive pulmonary disease (copd)" use karta hai
precautions_df["Disease"] = precautions_df["Disease"].replace(
    {"copd": "chronic obstructive pulmonary disease (copd)"}
)


In [11]:
# Description lookup: disease -> description text
description_lookup = dict(zip(description_df["Disease"], description_df["Description"]))

# Precaution lookup: disease -> {Precaution_1..4}
precaution_lookup = (
    precautions_df
    .set_index("Disease")[["Precaution_1", "Precaution_2", "Precaution_3", "Precaution_4"]]
    .to_dict("index")
)

# Diet lookup: disease -> list of diet items (stored as string list in csv, so eval karna padta hai)
diet_lookup = {
    row["Disease"]: ast.literal_eval(row["Diet"])
    for _, row in diets_df.iterrows()
}

# Workout lookup: disease -> list of workouts
workout_lookup = {
    row["Disease"]: ast.literal_eval(row["Workouts"])
    for _, row in workout_df.iterrows()
}

# Medication lookup: disease -> list of medications
medication_lookup = {
    row["Disease"]: ast.literal_eval(row["Medication"])
    for _, row in medications_df.iterrows()
}

print("Lookups ready:", len(description_lookup), "diseases covered")


Lookups ready: 100 diseases covered


## 10. Confidence Level Helper

In [12]:
def get_confidence_level(probability):
    if probability >= 0.70:
        return "HIGH"
    elif probability >= 0.40:
        return "MODERATE"
    else:
        return "LOW"


## 11. Final Prediction Function
Symptoms input karo, ye function top disease + probability + description + precautions + diet + workout + medications sab ek saath return karta hai.

In [13]:
def predict_disease_details(symptoms, top_n=5):

    # Step 1: har symptom ke liye 0 se start karo
    input_data = np.zeros(len(symptom_columns), dtype=int)

    # Step 2: user ke diye gaye symptoms ko 1 karo
    for symptom in symptoms:
        if symptom in symptom_columns:
            index = list(symptom_columns).index(symptom)
            input_data[index] = 1

    # Step 3: model ko same column names ke saath DataFrame chahiye
    input_df = pd.DataFrame([input_data], columns=symptom_columns)

    # Step 4: har disease ki probability nikalo
    probabilities = model.predict_proba(input_df)[0]

    # Step 5: sabse zyada probability wali top_n diseases
    top_indices = np.argsort(probabilities)[-top_n:][::-1]

    top_predictions = [
        {
            "disease": label_encoder.inverse_transform([i])[0],
            "probability": float(probabilities[i])
        }
        for i in top_indices
    ]

    # Step 6: best (rank 1) prediction ke details nikalo
    best = top_predictions[0]
    disease_key = best["disease"].strip().lower()

    return {
        "status": "success",
        "top_predictions": top_predictions,
        "best_disease": best["disease"],
        "confidence": best["probability"],
        "confidence_level": get_confidence_level(best["probability"]),
        "description": description_lookup.get(disease_key),
        "precautions": precaution_lookup.get(disease_key),
        "diet": diet_lookup.get(disease_key),
        "workout": workout_lookup.get(disease_key),
        "medications": medication_lookup.get(disease_key),
    }


## 12. Test It

In [14]:
test_symptoms = [
    "anxiety and nervousness",
    "depression",
    "insomnia"
]

result = predict_disease_details(test_symptoms)

print("Predicted Disease:", result["best_disease"])
print(f"Confidence: {result['confidence']:.2%} ({result['confidence_level']})")

print("\nTop predictions:")
for i, pred in enumerate(result["top_predictions"], start=1):
    print(f"  {i}. {pred['disease']} - {pred['probability']:.2%}")

print("\nDescription:", result["description"])

print("\nPrecautions:")
for value in (result["precautions"] or {}).values():
    print(" -", value)

print("\nDiet:")
for item in result["diet"] or []:
    print(" -", item)

print("\nWorkout:")
for item in result["workout"] or []:
    print(" -", item)

print("\nMedications:")
for item in result["medications"] or []:
    print(" -", item)


Predicted Disease: schizophrenia
Confidence: 30.91% (LOW)

Top predictions:
  1. schizophrenia - 30.91%
  2. panic disorder - 20.92%
  3. personality disorder - 19.12%
  4. depression - 13.73%
  5. anxiety - 13.38%

Description: Schizophrenia is a severe psychiatric disorder involving distortions in thinking, perception, emotions, language, and behavior, often with hallucinations or delusions.

Precautions:
 - Adhere to medication
 - Avoid substance abuse
 - Attend therapy sessions
 - Build a support network

Diet:
 - Omega-3 fatty acids (fish, flaxseeds)
 - Complex carbs (whole grains, vegetables)
 - Vitamin B-complex foods (eggs, nuts)
 - Antioxidant-rich foods (berries, citrus)
 - Limit caffeine and processed foods

Workout:
 - Structured group workouts: Promote social interaction
 - Walking or jogging: Boosts brain chemicals
 - Tai chi: Improves focus and calm
 - Avoid sensory overload: Choose quiet environments

Medications:
 - Antipsychotics (e.g., Risperidone, Olanzapine)
 - Clo

 ## 13. Confusing Disease Detection + Symptom Suggestions

In [20]:
## Phase 3 — Step 2 & 3: Confusing Disease Detection + Symptom Suggestion

def find_confusing_diseases(top_predictions, margin=0.15):
    """
    Top prediction ke probability ke 'margin' ke andar jo bhi diseases hain,
    unhe 'confusing' maana jayega (matlab model confuse ho raha hai in mein).
    """
    top_probability = top_predictions[0]["probability"]

    confusing = [
        pred["disease"] for pred in top_predictions
        if (top_probability - pred["probability"]) <= margin
    ]

    return confusing


def suggest_additional_symptoms(confusing_diseases, user_symptoms, top_k=5):
    """
    Confusing diseases ke actual data (df) se aise symptoms dhundo
    jo un diseases ko ek doosre se alag karte hain (differentiate karte hain).
    """

    # Step 1: har confusing disease ke symptoms ka frequency table banao
    disease_symptom_freq = {}

    for disease in confusing_diseases:
        disease_rows = df[df["diseases"].str.lower() == disease.lower()]
        disease_symptom_freq[disease] = disease_rows[symptom_columns].mean()

    # Step 2: har symptom ke liye — diseases ke beech frequency ka difference nikalo
    # Jitna zyada difference, utna zyada wo symptom differentiate karta hai
    symptom_scores = {}

    for symptom in symptom_columns:

        # Symptom pehle se user ne diya hai to skip karo
        if symptom in user_symptoms:
            continue

        freqs = [disease_symptom_freq[d][symptom] for d in confusing_diseases]
        differentiation_score = max(freqs) - min(freqs)

        # Sirf wahi symptoms lo jo kam se kam kisi ek disease mein commonly hote hain
        if differentiation_score > 0 and max(freqs) >= 0.4:
            symptom_scores[symptom] = differentiation_score

    # Step 3: top_k sabse zyada differentiating symptoms return karo
    sorted_symptoms = sorted(symptom_scores.items(), key=lambda x: x[1], reverse=True)

    return [symptom for symptom, score in sorted_symptoms[:top_k]]


def print_uncertain_result(result, user_symptoms):
    """
    LOW/MODERATE confidence hone par user-friendly warning + suggestions print karo.
    """
    print("⚠️  The prediction is uncertain.\n")

    print("Possible conditions:")
    for i, pred in enumerate(result["top_predictions"][:3], start=1):
        print(f"  {i}. {pred['disease'].title()} — {pred['probability']:.2%}")

    confusing_diseases = find_confusing_diseases(result["top_predictions"])
    suggestions = suggest_additional_symptoms(confusing_diseases, user_symptoms)

    print("\nTo improve the prediction, provide additional symptoms such as:")
    for symptom in suggestions:
        print(f"  - {symptom}")

## 14. Test It

In [21]:
test_symptoms = ["sharp abdominal pain", "vomiting", "nausea"]

result = predict_disease_details(test_symptoms)

if result["confidence_level"] == "HIGH":
    print("Predicted Disease:", result["best_disease"])
    print(f"Confidence: {result['confidence']:.2%}")
else:
    print_uncertain_result(result, test_symptoms)

⚠️  The prediction is uncertain.

Possible conditions:
  1. Esophagitis — 11.32%
  2. Problem During Pregnancy — 9.46%
  3. Acute Kidney Injury — 9.17%

To improve the prediction, provide additional symptoms such as:
  - retention of urine
  - pelvic pain
  - lower abdominal pain
  - back pain
  - problems during pregnancy


## 15. Re-Prediction With Additional Symptoms

In [22]:
def repredict_with_more_symptoms(original_symptoms, additional_symptoms):
    """
    Purane symptoms + naye symptoms combine karke dobara prediction karta hai.
    """
    # Dono lists ko combine karo, duplicate symptoms hata do
    combined_symptoms = list(set(original_symptoms + additional_symptoms))

    result = predict_disease_details(combined_symptoms)

    return combined_symptoms, result

## 16. Full Interactive Loop (jab tak HIGH confidence na aaye)

In [29]:
def interactive_prediction(initial_symptoms, max_rounds=3):
    """
    Predict karta hai, agar confidence LOW/MODERATE ho to symptoms suggest karta hai,
    user se naye symptoms leta hai, aur dubara predict karta hai — 
    jab tak HIGH confidence na aaye ya max_rounds khatam na ho jaayein.
    """
    current_symptoms = initial_symptoms.copy()

    for round_number in range(1, max_rounds + 1):

        print(f"\n{'='*50}")
        print(f"Round {round_number}")
        print(f"{'='*50}")

        result = predict_disease_details(current_symptoms)

        # HIGH confidence -> yahin ruk jao, poori recommendation dikhao
        if result["confidence_level"] == "HIGH":
            print_final_recommendation(result)
            return result

        # LOW/MODERATE -> uncertain result + suggestions dikhao
        print_uncertain_result(result, current_symptoms)

        # Max rounds khatam ho gaye to yahin ruk jao
        if round_number == max_rounds:
            print("\n⚠️ Max rounds reached. Best guess ke sath ruk rahe hain.")
            return result

        # User se naye symptoms lo (comma se separate karke)
        user_input = input("\nEnter additional symptoms (comma separated): ")
        new_symptoms = [s.strip() for s in user_input.split(",") if s.strip()]

        if not new_symptoms:
            print("Koi naya symptom nahi diya, ruk rahe hain.")
            return result

        current_symptoms, result = repredict_with_more_symptoms(current_symptoms, new_symptoms)

    return result

## 17. Final Recommendation Formatting

In [30]:
def print_final_recommendation(result):
    """
    HIGH confidence prediction ke liye poora final result — 
    description, precautions, diet, workout, medications — clean format mein print karta hai.
    """
    print("\n" + "=" * 50)
    print("PREDICTION")
    print("=" * 50)

    print(f"\nDisease: {result['best_disease'].title()}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"Confidence Level: {result['confidence_level']}")

    print("\nDescription:")
    print(result["description"])

    print("\nPrecautions:")
    for value in (result["precautions"] or {}).values():
        print(" -", value)

    print("\nDiet:")
    for item in result["diet"] or []:
        print(" -", item)

    print("\nWorkout:")
    for item in result["workout"] or []:
        print(" -", item)

    print("\nMedications:")
    for item in result["medications"] or []:
        print(" -", item)

    print("\n" + "=" * 50)

## 18. Test It

In [31]:
final_result = interactive_prediction(["sharp abdominal pain", "vomiting", "nausea"])


Round 1
⚠️  The prediction is uncertain.

Possible conditions:
  1. Esophagitis — 11.32%
  2. Problem During Pregnancy — 9.46%
  3. Acute Kidney Injury — 9.17%

To improve the prediction, provide additional symptoms such as:
  - retention of urine
  - pelvic pain
  - lower abdominal pain
  - back pain
  - problems during pregnancy



Enter additional symptoms (comma separated):  retention of urine, pelvic pain



Round 2
⚠️  The prediction is uncertain.

Possible conditions:
  1. Acute Kidney Injury — 43.80%
  2. Chronic Constipation — 24.50%
  3. Problem During Pregnancy — 18.15%

To improve the prediction, provide additional symptoms such as:



Enter additional symptoms (comma separated):  back pain



Round 3
⚠️  The prediction is uncertain.

Possible conditions:
  1. Problem During Pregnancy — 53.43%
  2. Pelvic Inflammatory Disease — 25.70%
  3. Acute Kidney Injury — 9.84%

To improve the prediction, provide additional symptoms such as:

⚠️ Max rounds reached. Best guess ke sath ruk rahe hain.
